In [56]:
enter_long_limit = -0.005
exit_long_limit = 0.00

enter_short_limit = 0.005   
exit_short_limit = 0.00  

dollar_constant = 10000

commission_per_shs = -0.005


In [57]:
import numpy as np
import pandas as pd

In [58]:
df = pd.read_csv("converts.csv")
df = df.dropna(how="all")
df = df.dropna(axis=1, how="all")
df = df.reset_index(drop=True)

In [59]:
df['DATE'] = pd.to_datetime(df['DATE'])
for col in ["CWB", "ICVT"]:
    df[col] = pd.to_numeric(
        df[col]
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

df = df.rename(columns={"CWB": "CWB_price", "ICVT": "ICVT_price"})

In [60]:
df['CWB_ratio_chg'] = df['CWB_price'].pct_change()
df['ICVT_ratio_chg'] = df['ICVT_price'].pct_change()
df['CWB_price_chg'] = df['CWB_price'].diff()
df['ICVT_price_chg'] = df['ICVT_price'].diff()
df['ICVT/CWB'] = df['ICVT_price'] / df['CWB_price']
df['ICVT/CWB_moving_avg'] = df['ICVT/CWB'].rolling(window=20).mean()
df['CWB*_price'] = df['CWB_price'] * df['ICVT/CWB_moving_avg'].shift(1)
df['LN(CWB*/ICVT)'] = np.log(df['CWB*_price'] / df['ICVT_price'])

In [61]:
df['enter_long'] = df['LN(CWB*/ICVT)'] < enter_long_limit
df['exit_long'] = df['LN(CWB*/ICVT)'] > exit_long_limit

df['enter_short'] = df['LN(CWB*/ICVT)'] > enter_short_limit
df['exit_short'] = df['LN(CWB*/ICVT)'] < exit_short_limit   


In [62]:
df["current_position"] = 0
df["target_position"] = 0

In [63]:
for i in range(len(df)):

    # Current position is yesterday's target position
    current_position = 0 if i == 0 else df.at[i - 1, "target_position"]
    df.at[i, "current_position"] = current_position

    # Default: keep the existing position
    target_position = current_position

    if current_position == 0:
        if df.at[i, "enter_long"]:
            target_position = 1
        elif df.at[i, "enter_short"]:
            target_position = -1

    elif current_position == 1:
        # Reverse directly from long to short
        if df.at[i, "enter_short"]:
            target_position = -1
        elif df.at[i, "exit_long"]:
            target_position = 0

    elif current_position == -1:
        # Reverse directly from short to long
        if df.at[i, "enter_long"]:
            target_position = 1
        elif df.at[i, "exit_short"]:
            target_position = 0

    df.at[i, "target_position"] = target_position    

In [ ]:
df['CWB_current_shs'] = 0
df['CWB_current_inv'] = 0
df['CWB_target_shs'] = 0
df['CWB_shs_to_trade'] = 0

df['ICVT_current_shs'] = 0
df['ICVT_current_inv'] = 0
df['ICVT_target_shs'] = 0
df['ICVT_shs_to_trade'] = 0

In [ ]:
df['ICVT_target_shs'] = round(-df['target_position'] * dollar_constant / df['ICVT_price'])
df['CWB_target_shs'] = round(-df['ICVT_target_shs'] * df['ICVT/CWB_moving_avg'].shift(1).fillna(0))

df['CWB_current_shs'] = df['CWB_target_shs'].shift(1).fillna(0)
df['ICVT_current_shs'] = df['ICVT_target_shs'].shift(1).fillna(0)    

df['CWB_current_inv'] = df['CWB_current_shs'] * df['CWB_price'].shift(1).fillna(0)
df['ICVT_current_inv'] = df['ICVT_current_shs'] * df['ICVT_price'].shift(1).fillna(0)   

df['CWB_shs_to_trade'] = df['CWB_target_shs'] - df['CWB_current_shs']
df['ICVT_shs_to_trade'] = df['ICVT_target_shs'] - df['ICVT_current_shs']

In [66]:
df['CWB_daily_profit'] = df['CWB_current_shs'] * df['CWB_price_chg']
df['ICVT_daily_profit'] = df['ICVT_current_shs'] * df['ICVT_price_chg']
df['daily_profit'] = df['CWB_daily_profit'] + df['ICVT_daily_profit']
df['profit_cum_sum'] = df['daily_profit'].cumsum() 

In [67]:
df['CWB_commission'] = df['CWB_shs_to_trade'].abs() * commission_per_shs
df['ICVT_commission'] = df['ICVT_shs_to_trade'].abs() * commission_per_shs
df['daily_commission'] = df['CWB_commission'] + df['ICVT_commission']
df['commission_cum_sum'] = df['daily_commission'].cumsum()

In [68]:
df['daily_net_profit'] = df['daily_profit'] + df['daily_commission']
df['net_profit_cum_sum'] = df['daily_net_profit'].cumsum()
df['max_cum_sum'] = df['net_profit_cum_sum'].cummax()
df['drawdown'] = df['net_profit_cum_sum'] - df['max_cum_sum']

In [69]:
df.to_csv("converts results.csv", index=False)

In [70]:
df

,DATE,CWB_price,ICVT_price,CWB_ratio_chg,ICVT_ratio_chg,CWB_price_chg,ICVT_price_chg,ICVT/CWB,ICVT/CWB_moving_avg,CWB*_price,...,daily_profit,profit_cum_sum,CWB_commission,ICVT_commission,daily_commission,commission_cum_sum,daily_net_profit,net_profit_cum_sum,max_cum_sum,drawdown
0,2025-01-02,78.00,85.02,NaN,NaN,NaN,NaN,1.090000,NaN,NaN,...,NaN,NaN,NaN,-0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-03,79.02,86.29,0.013077,0.014938,1.02,1.27,1.092002,NaN,NaN,...,0.0,0.00,NaN,-0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-01-06,79.53,86.82,0.006454,0.006142,0.51,0.53,1.091664,NaN,NaN,...,0.0,0.00,NaN,-0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-01-07,78.77,85.90,-0.009556,-0.010597,-0.76,-0.92,1.090517,NaN,NaN,...,-0.0,0.00,NaN,-0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-01-08,78.48,85.58,-0.003682,-0.003725,-0.29,-0.32,1.090469,NaN,NaN,...,-0.0,0.00,NaN,-0.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234,2025-12-09,90.80,100.23,0.000661,0.000599,0.06,0.06,1.103855,1.106759,100.510509,...,0.0,192.51,-0.0,-0.0,-0.0,-12.3,0.0,180.21,250.83,-70.62
235,2025-12-10,91.02,100.26,0.002423,0.000299,0.22,0.03,1.101516,1.106373,100.737176,...,0.0,192.51,-0.0,-0.0,-0.0,-12.3,0.0,180.21,250.83,-70.62
236,2025-12-11,91.41,100.79,0.004285,0.005286,0.39,0.53,1.102615,1.106051,101.133577,...,0.0,192.51,-0.0,-0.0,-0.0,-12.3,0.0,180.21,250.83,-70.62
237,2025-12-12,90.26,99.53,-0.012581,-0.012501,-1.15,-1.26,1.102703,1.105782,99.832119,...,0.0,192.51,-0.0,-0.0,-0.0,-12.3,0.0,180.21,250.83,-70.62
